# Multi-Armed Bandits: learn action values without states

A stationary bandit estimates the mean reward of each arm with the incremental sample average

$$Q_{n+1}(a)=Q_n(a)+\frac{1}{N_{n+1}(a)}\left[R_{n+1}-Q_n(a)\right].$$

Here $Q_n(a)$ is arm $a$'s estimate after $n$ pulls, $R_{n+1}$ the new reward, and $N_{n+1}(a)$ its updated pull count. An $\varepsilon$-greedy policy explores randomly with probability $\varepsilon$ and otherwise pulls the current best arm.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

TOTAL_PULLS = 5_000
EPSILON_START = 0.2
EPSILON_END = 0.05
EXPLORATION_STEPS = 2_000
SEED = 7


class GaussianBandit(gym.Env):
    observation_space = gym.spaces.Discrete(1)
    action_space = gym.spaces.Discrete(5)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        return 0, {}

    def step(self, action):
        means = np.arange(self.action_space.n) * 0.25
        reward = self.np_random.normal(means[action], 1.0)
        return 0, float(reward), True, False, {"optimal_action": int(means.argmax())}


env = GaussianBandit()
rng = np.random.default_rng(SEED)
q_values = np.zeros(env.action_space.n)
action_counts = np.zeros(env.action_space.n, dtype=np.int64)

## 1. Explore and update

The linear schedule is

$$\varepsilon_t=\varepsilon_{start}+\min(t/T,1)(\varepsilon_{end}-\varepsilon_{start}),$$

where $T$ is `EXPLORATION_STEPS`. Each observed reward updates only the selected arm.

In [ ]:
def epsilon_at(step):
    fraction = min(step / EXPLORATION_STEPS, 1.0)
    return EPSILON_START + fraction * (EPSILON_END - EPSILON_START)


def select_action(step, deterministic=False):
    if not deterministic and rng.random() < epsilon_at(step):
        return int(rng.integers(len(q_values)))
    return int(np.argmax(q_values))


def update(action, reward):
    action_counts[action] += 1
    step_size = 1 / action_counts[action]
    q_values[action] += step_size * (reward - q_values[action])


rewards = []
optimal_actions = []
env.reset(seed=SEED)
for step in range(TOTAL_PULLS):
    action = select_action(step)
    _, reward, _, _, info = env.step(action)
    update(action, reward)
    rewards.append(reward)
    optimal_actions.append(action == info["optimal_action"])
env.close()

## 2. Inspect reward and learned values

The moving-average reward should approach the best arm's true mean, while the value bars show the estimates learned for every arm.

In [ ]:
window = 100
moving_reward = np.convolve(rewards, np.ones(window) / window, mode="valid")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(np.arange(window - 1, TOTAL_PULLS), moving_reward)
axes[0].axhline(1.0, color="black", linestyle="--", label="Best true mean")
axes[0].set(xlabel="Pull", ylabel="Reward", title="Moving-average reward")
axes[0].legend()
axes[1].bar(np.arange(len(q_values)), q_values)
axes[1].set(xlabel="Arm", ylabel="Estimated mean reward", title="Learned arm values")
for axis in axes:
    axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 3. Evaluate the greedy arm

A bandit has no visual state to render. This uses a fresh bandit for 1,000 independent pulls of the learned best arm.

In [ ]:
evaluation_env = GaussianBandit()
evaluation_env.reset(seed=10)
evaluation_rewards = []
try:
    for _ in range(1_000):
        action = select_action(step=TOTAL_PULLS, deterministic=True)
        _, reward, _, _, _ = evaluation_env.step(action)
        evaluation_rewards.append(reward)
finally:
    evaluation_env.close()

print("Greedy arm:", select_action(TOTAL_PULLS, deterministic=True))
print(f"Mean reward: {np.mean(evaluation_rewards):.2f} +/- {np.std(evaluation_rewards):.2f}")